# Stochastic Calculus from Zero · Episode 6: Martingales: fair games

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The coin world.** A share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses, eight possible worlds. Heads has a 60% real-world chance; a deposit grows 2% a step. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The martingale test, node by node · 4. The option's value is a martingale too · 5. Copying the option · 6. Constant average is not enough · 7. History · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 6,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

With a 50% coin each toss multiplies the price by $0.5 \times 1.2 + 0.5 \times 0.8 = 1$ on average, so after a first head the best guess of $S_3$ is $S_1 = 120$: a process with no drift.

In [ ]:
out["prev"] = {"answer": 120.0}

## 3. The martingale test, node by node

A process $M$ is a **martingale** if at every node your best guess of the next step is where you are now: $E[M_{n+1} \mid \mathcal{F}_n] = M_n$.

Test the **discounted** share $S_n / 1.02^n$ (a deposit grows 2% a step) at all 7 nodes of the three-toss tree (1 + 2 + 4: the tree doesn't merge HT and TH). Under the real-world odds $p = 0.6$ it drifts up by $1.04 / 1.02 = 1.0196$ a step: not a martingale. Under $q = (1.02 - 0.8)/(1.2 - 0.8) = 0.55$ it is a martingale at every node.

In [ ]:
cw = CoinWorld()
R = 1 + cw.rate
nodes = []
for n in range(cw.steps):
    for b in sorted({w[:n] for w in cw.worlds}):
        now = (cw.price(b, n) if b else cw.s0) / R ** n
        nxt = lambda w, n=n: cw.price(w, n + 1) / R ** (n + 1)
        nodes.append({"node": b or "start", "now": now, "next_P": cw.given(n, nxt, "P")[b], "next_Q": cw.given(n, nxt, "Q")[b]})
assert len(nodes) == 7
assert all(abs(x["next_Q"] - x["now"]) < 1e-9 for x in nodes)
assert all(x["next_P"] > x["now"] for x in nodes)
out["mart"] = {"nodes": nodes, "q": cw.q, "p": cw.p, "drift_P": (cw.p * cw.up + (1 - cw.p) * cw.down) / R,
               "n_nodes": len(nodes), "start_next_P": nodes[0]["next_P"], "start_next_Q": nodes[0]["next_Q"]}
pd.DataFrame(nodes)

## 4. The option's value is a martingale too

The call struck at 100 has value $V_n = E_Q[\text{call} \mid \mathcal{F}_n] / 1.02^{3-n}$ at each node, and $V_n / 1.02^n$ is a $Q$-martingale. Its value today is the price.

In [ ]:
K = 100.0
call = lambda w: max(cw.price(w, 3) - K, 0.0)
V = {n: {b: v / R ** (cw.steps - n) for b, v in cw.given(n, call, "Q").items()} for n in range(cw.steps + 1)}
for n in range(cw.steps):
    for b in V[n]:
        assert abs(V[n][b] - (cw.q * V[n + 1][b + "H"] + (1 - cw.q) * V[n + 1][b + "T"]) / R) < 1e-9
rows = cw.table(call=call, V2=lambda w: V[2][w[:2]], V1=lambda w: V[1][w[:1]])
out["coin"] = {"worlds": rows}
out["option"] = {"price": V[0][""], "V1": V[1], "V2": V[2], "strike": K,
                 "naive_P": cw.expect(call, "P") / R ** 3}
out["option"]

## 5. Copying the option

Hold $\Delta_0 = \dfrac{V_1(H) - V_1(T)}{S_1(H) - S_1(T)}$ shares and borrow the rest (Series 3, Episode 5). After one toss the portfolio is worth exactly $V_1$ in both blocks: the martingale price is the cost of copying the option.

In [ ]:
delta = (V[1]["H"] - V[1]["T"]) / (cw.price("H", 1) - cw.price("T", 1))
cash = V[0][""] - delta * cw.s0
up, down = delta * cw.price("H", 1) + cash * R, delta * cw.price("T", 1) + cash * R
assert abs(up - V[1]["H"]) < 1e-9 and abs(down - V[1]["T"]) < 1e-9
out["copy"] = {"delta": delta, "borrow": -cash, "after_H": up, "after_T": down,
               "table": [{"what": "Shares held", "value": delta}, {"what": "Borrowed", "value": -cash},
                         {"what": "Worth after heads", "value": up}, {"what": "Worth after tails", "value": down}]}
out["copy"]["table"]

## 6. Constant average is not enough

Let $X_1 = +1$ if the first toss is heads, $-1$ if tails, and $X_2 = -X_1$. Under a fair coin both average 0 at every time, but after the first toss your best guess of $X_2$ is $-X_1$, not $X_1$: not a martingale.

In [ ]:
fair = CoinWorld(p=0.5)
X1 = lambda w: 1.0 if w[0] == "H" else -1.0
X2 = lambda w: -X1(w)
guess = fair.given(1, X2)
assert fair.expect(X1) == 0 and fair.expect(X2) == 0 and guess["H"] == -1 and guess["T"] == 1
out["counter"] = {"mean": 0, "guess_after_H": guess["H"], "X1_after_H": 1}

## 7. History

"Martingale" was first the name of a family of betting strategies (doubling up after each loss). Jean Ville brought the word into probability theory in his 1939 thesis *Étude critique de la notion de collectif*; much of the theory was then developed by Joseph Doob.

In [ ]:
out["history"] = {"ville": 1939}

## 8. This episode's question

Under $q$, is the undiscounted share a martingale? No: $E_Q[S_{n+1} \mid \mathcal{F}_n] = 1.02\, S_n$, it grows like the deposit. Only after discounting does the drift disappear.

In [ ]:
growth_Q = cw.q * cw.up + (1 - cw.q) * cw.down
assert abs(growth_Q - R) < 1e-12
out["question"] = {"growth_Q": growth_Q, "rate_pct": cw.rate * 100}

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Check the $Q$-martingale property of the discounted share at node HT.

<details><summary>Answer</summary>

At HT, $S_2 = 96$, discounted $96 / 1.02^2$. Next step: $0.55 \times 115.2 + 0.45 \times 76.8 = 63.36 + 34.56 = 97.92 = 1.02 \times 96$, discounted by $1.02^3$: $96 / 1.02^2$. ✓

</details>

**Exercise 2 (By hand).** Is the number of heads minus $0.6n$ a $P$-martingale?

<details><summary>Answer</summary>

Yes: each toss adds 1 with chance 0.6 and 0 otherwise, so the expected increase is 0.6, exactly what $0.6(n+1) - 0.6n$ subtracts.

</details>

**Exercise 3 (By hand).** In the counterexample, compute $E[X_2 \mid \mathcal{F}_1]$ in both blocks.

<details><summary>Answer</summary>

After H, $X_2 = -1$ for sure; after T, $X_2 = +1$. So $E[X_2 \mid \mathcal{F}_1] = -X_1 \neq X_1$.

</details>

**Exercise 4 (In code).** Check that the discounted call value $V_n / 1.02^n$ is a $Q$-martingale at every node, and **not** a $P$-martingale.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
for n in range(cw.steps):
    for b in V[n]:
        q_next = (cw.q * V[n + 1][b + "H"] + (1 - cw.q) * V[n + 1][b + "T"]) / R ** (n + 1)
        p_next = (cw.p * V[n + 1][b + "H"] + (1 - cw.p) * V[n + 1][b + "T"]) / R ** (n + 1)
        assert abs(q_next - V[n][b] / R ** n) < 1e-9
        assert p_next > V[n][b] / R ** n or V[n][b] == 0
print("discounted call: Q-martingale at every node; drifts up under P")

**Exercise 5 (In code).** If the deposit grew 3% a step instead of 2%, what $q$ makes the discounted share a martingale?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
cw3 = CoinWorld(rate=0.03)
assert abs(cw3.q - 0.575) < 1e-12
assert abs(cw3.q * 1.2 + (1 - cw3.q) * 0.8 - 1.03) < 1e-12
print(f"q = {cw3.q:.3f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")